# 05 Colab NIFTY50 Batch Pipeline — Full Run

Runs the full `data/raw/nifty50/` CSV corpus. Cells 1 and 2 (raw cache and feature engineering) are **skipped by default** because all parquets are already built. Set `SKIP_IF_EXISTS = False` in cell 1 or 2 only if you need to re-process specific symbols.

**Run order for a full training run (all parquets already on Drive):**
1. Setup cell — mount Drive, install package
2. Git pull cell — get latest code
3. LightGBM install cell — build CUDA-enabled LightGBM
4. Config cell — verify paths and config
5. *(Optional)* Cell 1 — only if any raw CSVs have not yet been cached as parquet
6. *(Optional)* Cell 2 — only if any processed feature parquets are missing
7. **Cell 3 — full combined training (run this)**
8. Archive cell — zip and save outputs


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import gc
import subprocess
import sys

BASE = Path('/content/drive/MyDrive/trading_system')
REPO_DIR = BASE / 'TradingBot26'
DATA_DIR = BASE / 'data'
MODEL_DIR = BASE / 'models'
LOG_DIR = BASE / 'logs'
REPORT_DIR = BASE / 'reports'
ARTIFACT_DIR = BASE / 'artifacts'

for path in (DATA_DIR / 'raw', DATA_DIR / 'processed', MODEL_DIR, LOG_DIR, REPORT_DIR, ARTIFACT_DIR):
    path.mkdir(parents=True, exist_ok=True)

if not (REPO_DIR / 'pyproject.toml').exists():
    raise FileNotFoundError(f'Copy or clone this repo to {REPO_DIR} before running this notebook')

%cd $REPO_DIR
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)])

In [ ]:
!git -C /content/drive/MyDrive/trading_system/TradingBot26 pull


In [ ]:
import shutil
import site
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'lightgbm'], check=False)
for site_dir in site.getsitepackages():
    for path in Path(site_dir).glob('lightgbm*'):
        if path.is_dir():
            shutil.rmtree(path)
        elif path.exists():
            path.unlink()

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '--no-cache-dir',
    '--force-reinstall',
    '--no-binary=lightgbm',
    '--config-settings=cmake.define.USE_CUDA=ON',
    'lightgbm>=4.3',
])

import lightgbm
from lightgbm import LGBMClassifier
from lightgbm.basic import _LIB
print('LightGBM:', lightgbm.__version__)
print('Native library:', _LIB._name)
probe = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    n_estimators=1,
    min_data_in_leaf=1,
    min_data_in_bin=1,
    device_type='cuda',
    max_bin=63,
    verbosity=1,
)
probe.fit([[0, 0], [0, 1], [1, 0], [1, 1], [2, 0], [2, 1]], [0, 1, 2, 0, 1, 2])
print('LightGBM CUDA smoke test passed.')
# NOTE: If this probe.fit() fails with a CUDA error, set LGBM_DEVICE_TYPE = 'cpu'
# in the training cell below. This happens on V100 runtimes where the CUDA build
# compiles but the GPU device_type probe fails at runtime.

In [ ]:
import importlib
from dataclasses import fields, replace

import pandas as pd

repo_path = str(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
for module_name in ('config', 'data.loader'):
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()

import config as config_module
from config import MarketConfig, PathConfig
from data.loader import MarketDataLoader, standardize_ohlcv

market_fields = {field.name for field in fields(MarketConfig)}
required_fields = {'local_intraday_path', 'local_intraday_pattern'}
if not required_fields.issubset(market_fields):
    raise RuntimeError(
        'Colab is importing a stale TradingBot26 config.py from '
        f'{config_module.__file__}. Pull the latest repo commit, rerun the install cell, '
        'then use Runtime > Restart runtime before rerunning this notebook.'
    )
print(f'Using TradingBot26 config from: {config_module.__file__}')

paths = PathConfig(
    root=BASE,
    raw_data_dir=DATA_DIR / 'raw',
    processed_data_dir=DATA_DIR / 'processed',
    artifact_dir=ARTIFACT_DIR,
    model_artifact_dir=MODEL_DIR,
    report_dir=REPORT_DIR,
)

## 1. Cache raw intraday files *(skip if already done)*

This cell is **safe to skip** if all raw CSVs have already been converted to per-symbol 5-minute Parquet files in `data/raw/`. It is fully idempotent — `FORCE_REFRESH = False` means any already-cached parquet is left untouched.

**When to run this cell:**
- First time setup on a fresh Drive, or
- You have added new raw CSV files since the last run.

**When to skip:**
- All `*_5m_intraday.parquet` files already exist in `data/raw/` for every symbol you want to train on.


In [ ]:
# ── SKIP GUARD ────────────────────────────────────────────────────────────────
# Set to False only if you have new raw CSVs that have not yet been cached.
RUN_RAW_CACHE = False
# ──────────────────────────────────────────────────────────────────────────────

if not RUN_RAW_CACHE:
    existing = list((DATA_DIR / 'raw').glob('*_5m_intraday.parquet'))
    print(f'Raw cache cell skipped. Found {len(existing)} existing intraday parquet(s) in data/raw/.')
    print('Set RUN_RAW_CACHE = True in this cell if you need to process new CSV files.')
else:
    import sys
    repo_path = str(REPO_DIR)
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

    from scripts.csv_discovery import discover_intraday_csv_files, symbol_from_intraday_csv_path

    RAW_INTRADAY_SOURCE_DIRS = [
        DATA_DIR / 'raw' / 'nifty50stocks',
        DATA_DIR / 'raw' / '18',
        DATA_DIR / 'raw' / 'nifty50',
        REPO_DIR / 'data' / 'raw' / 'nifty50stocks',
        REPO_DIR / 'data' / 'raw' / '18',
        REPO_DIR / 'data' / 'raw' / 'nifty50',
    ]
    RAW_INTRADAY_CSV_FILES = discover_intraday_csv_files(RAW_INTRADAY_SOURCE_DIRS)

    if not RAW_INTRADAY_CSV_FILES:
        print('No supported intraday CSV files found under the Colab raw folders.')
    else:
        FORCE_REFRESH = False  # True only to re-cache already existing parquets
        MAX_FILES = 0          # 0 = no limit; set to N for a partial run
        processed = []
        print(f'Discovered {len(RAW_INTRADAY_CSV_FILES)} intraday CSV files.')
        for csv_path in RAW_INTRADAY_CSV_FILES:
            base_name = symbol_from_intraday_csv_path(csv_path)
            if base_name is None:
                print(f'Skipping unsupported filename: {csv_path.name}')
                continue
            market = MarketConfig(
                symbol=base_name,
                ticker=base_name,
                series='IDX' if base_name.upper().startswith(('NIFTY', 'BANK', 'INDIA VIX')) else 'EQ',
                interval='5m',
                intraday_source='local-csv',
                daily_source='intraday-resample',
                local_intraday_path=csv_path,
                lookback_days=55,
            )
            loader = MarketDataLoader(paths, market)
            intraday_result = loader.download_intraday(force_refresh=FORCE_REFRESH, source='local-csv')
            daily_result = loader.download_daily_context(force_refresh=FORCE_REFRESH)
            print(
                f"{csv_path.name} -> intraday={intraday_result.rows} rows, daily={daily_result.rows} rows, "
                f"output={intraday_result.path.name}"
            )
            processed.append((csv_path.name, intraday_result.path, daily_result.path))
            if MAX_FILES and len(processed) >= MAX_FILES:
                break
        print(f'Completed raw download for {len(processed)} intraday CSV files.')

## 2. Build features *(skip if already done)*

This cell is **safe to skip** if all processed feature parquets already exist in `data/processed/`. The pipeline checks for existing files and skips them when `FORCE_REFRESH = False`.

**When to run this cell:**
- First time setup, or
- You have added new symbols (new raw parquets) since the last feature build, or
- You have changed label/feature/normalizer config and want to rebuild.

**When to skip:**
- All `*_5m_features.parquet` files already exist in `data/processed/` for every symbol.


In [ ]:
# ── SKIP GUARD ────────────────────────────────────────────────────────────────
# Set to False only if you have new/missing processed parquets to build.
RUN_FEATURE_BUILD = False
# ──────────────────────────────────────────────────────────────────────────────

if not RUN_FEATURE_BUILD:
    existing = list((DATA_DIR / 'processed').glob('*_5m_features.parquet'))
    print(f'Feature build cell skipped. Found {len(existing)} existing feature parquet(s) in data/processed/.')
    print('Set RUN_FEATURE_BUILD = True in this cell if you need to build features for new symbols.')
else:
    from config import FeatureConfig, LabelConfig, MarketConfig, NormalizerConfig, PathConfig
    from data.loader import MarketDataLoader
    from features.pipeline import FeatureEngineeringPipeline

    RAW_INTRADAY_FILES = sorted(
        {
            path
            for path in (DATA_DIR / 'raw').glob('*_5m_intraday.parquet')
            if path.is_file() and not path.name.startswith('.')
        }
    )

    if not RAW_INTRADAY_FILES:
        print('No 5-minute intraday Parquet files found under data/raw.')
    else:
        FORCE_REFRESH = False  # True only to force-rebuild already existing feature parquets
        MAX_FILES = 0          # 0 = no limit; set to N for a partial run
        processed = []
        labels = LabelConfig(horizon=15, target_pct=0.005, stop_pct=0.003)
        normalizer = NormalizerConfig(window=200, min_periods=200)
        features = FeatureConfig(lag_periods=(1, 5, 10), volume_confirm_window=20, include_daily_context=True)
        for intraday_path in RAW_INTRADAY_FILES:
            base_name = intraday_path.stem.removesuffix('_5m_intraday')
            market = MarketConfig(
                symbol=base_name,
                ticker=base_name,
                series='IDX' if base_name.upper().startswith(('NIFTY', 'BANK')) else 'EQ',
                interval='5m',
                intraday_source='cached-parquet',
                daily_source='intraday-resample',
            )
            loader = MarketDataLoader(paths, market)
            processed_path = FeatureEngineeringPipeline(
                paths=paths,
                market=market,
                features=features,
                labels=labels,
                normalizer=normalizer,
            ).processed_path()
            if processed_path.exists() and not FORCE_REFRESH:
                print(f'Skipping {intraday_path.name} because {processed_path.name} already exists.')
                processed.append((intraday_path.name, processed_path))
                if MAX_FILES and len(processed) >= MAX_FILES:
                    break
                continue
            daily = loader.load_daily_context() if loader.daily_path().exists() else loader.download_daily_context(force_refresh=False)
            dataset = FeatureEngineeringPipeline(
                paths=paths,
                market=market,
                features=features,
                labels=labels,
                normalizer=normalizer,
            ).run(loader.load_intraday(), daily)
            print(
                f"{intraday_path.name} -> processed={len(dataset.frame):,} rows, features={len(dataset.feature_columns):,}, "
                f"output={dataset.feature_config_path.name}"
            )
            processed.append((intraday_path.name, dataset.feature_config_path))
            if MAX_FILES and len(processed) >= MAX_FILES:
                break
        print(f'Completed feature engineering for {len(processed)} files.')

## 3. Train combined models — full run

Loads all processed feature files from Drive (or local SSD if `STAGE_PROCESSED_TO_LOCAL = True`), builds one calendar-date 70/15/15 split across the full universe, trains pooled LightGBM, LSTM, and GRU.

**Key changes from the smoke-test run:**
- `MAX_FILES = 0` — no limit, uses every available processed parquet
- `SCOPE = 'nifty50_combined_5m_full'` — new scope avoids resuming from the 5-instrument smoke-test checkpoints
- `BATCH_SIZE = 2048` — matches what the trainer auto-selected in the last run, set explicitly
- `NUM_WORKERS = 4` — DataLoader prefetches batches in background so GPU stays fed
- `EPOCHS = 50` — sufficient for full-universe convergence; `_best.pt` is saved automatically at the optimal epoch
- `LEARNING_RATE = 3e-4` — more conservative than 1e-3 for stable multi-epoch training
- `STAGE_PROCESSED_TO_LOCAL = True` — copies parquets to /content SSD once, avoids repeated Drive FUSE reads
- `MIN_ROWS = 5000` — filters out symbols with fewer than ~2 years of 5-minute bars
- `LOG_EVERY_BATCHES = 50` — finer visibility on the ~1950+ batch-per-epoch full run
- `LGBM_DEVICE_TYPE = 'cuda'` — set to 'cpu' if the LightGBM CUDA probe above failed


In [ ]:
from pathlib import Path
import gc
import importlib
import json
import shutil
import sys
import numpy as np
import pandas as pd
import torch

repo_path = str(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
for module_name in (
    'models.torch_training',
    'models.torch_models',
    'scripts.train_pooled_lightgbm',
):
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()

from config import SplitConfig
from features.sequences import SequenceBuilder, SequenceDatasetArrays
from models.torch_models import GRUClassifier, LSTMClassifier
from models.torch_training import result_to_dict, train_torch_classifier_from_frames, write_torch_training_metadata
from scripts.train_pooled_lightgbm import (
    combine_instruments,
    date_based_split,
    load_instruments,
    shared_feature_columns,
    stage_processed_feature_files,
    train_pooled_lgbm_from_splits,
)

# ── FULL-RUN CONFIGURATION ────────────────────────────────────────────────────
LOOKBACK               = 60       # sequence lookback in bars; changing this invalidates existing checkpoints
EPOCHS                 = 50       # train for up to 50 epochs; _best.pt saves the optimal checkpoint automatically
BATCH_SIZE             = 2048     # explicitly set to what the trainer auto-used in the smoke test
LEARNING_RATE          = 3e-4     # reduced from 1e-3 for more stable full-universe convergence
NUM_WORKERS            = 4        # background DataLoader workers; set to 0 only on T4 (too little RAM)
LOG_EVERY_BATCHES      = 50       # log every 50 batches — ~1950+ batches/epoch on full universe
TRAIN_LIGHTGBM         = True
TRAIN_LSTM             = True
TRAIN_GRU              = True
REQUIRE_CUDA_FOR_TORCH = True     # hard-fail if no GPU — never run LSTM/GRU on CPU
MAX_FILES              = 0        # 0 = no limit; loads every *_5m_features.parquet in processed_root
LGBM_DEVICE_TYPE       = 'cuda'   # 'cuda' for A100/L4; set to 'cpu' if the CUDA probe above failed (V100)
LGBM_GPU_DEVICE_ID     = 0
MIN_ROWS               = 5000     # skip instruments with fewer than ~2 years of 5-minute bars
STAGE_PROCESSED_TO_LOCAL    = True   # copy parquets to /content SSD before training — critical for Drive FUSE speed
FORCE_REFRESH_LOCAL_STAGE   = False  # True only to force re-copy already staged files
CLEAN_LOCAL_STAGE_WHEN_DISABLED = False  # irrelevant when STAGE_PROCESSED_TO_LOCAL=True
LOCAL_PROCESSED_DIR    = Path('/content/trading_system_training/processed')
SCOPE                  = 'nifty50_combined_5m_full'  # NEW scope — avoids resuming from 5-instrument smoke-test checkpoints
RANDOM_SEED            = 42
# ─────────────────────────────────────────────────────────────────────────────

def log(message):
    print(message, flush=True)


def log_resources(stage):
    parts = [stage]
    try:
        import psutil
        process_ram = psutil.Process().memory_info().rss / (1024 ** 3)
        system_ram = psutil.virtual_memory()
        parts.append(
            f'process_ram={process_ram:.2f}GB system_used={system_ram.used / (1024 ** 3):.2f}/'
            f'{system_ram.total / (1024 ** 3):.2f}GB'
        )
    except Exception as exc:
        parts.append(f'ram=unavailable({exc})')
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / (1024 ** 3)
        reserved = torch.cuda.memory_reserved() / (1024 ** 3)
        max_allocated = torch.cuda.max_memory_allocated() / (1024 ** 3)
        parts.append(f'cuda_alloc={allocated:.2f}GB cuda_reserved={reserved:.2f}GB cuda_max={max_allocated:.2f}GB')
        try:
            free, total = torch.cuda.mem_get_info()
            parts.append(f'cuda_free={free / (1024 ** 3):.2f}/{total / (1024 ** 3):.2f}GB')
        except Exception as exc:
            parts.append(f'cuda_free=unavailable({exc})')
    else:
        parts.append('cuda=unavailable')
    log(' | '.join(parts))


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
log(f'Training cell started. device={device}')
log_resources('Initial resources')
if REQUIRE_CUDA_FOR_TORCH and (TRAIN_LSTM or TRAIN_GRU) and device.type != 'cuda':
    raise RuntimeError(
        'CUDA GPU is required for LSTM/GRU training. In Colab, choose Runtime > Change runtime type > GPU, then rerun setup.'
    )


def build_symbol_sequences(frame, *, feature_columns, lookback):
    builder = SequenceBuilder(lookback=lookback)
    xs, ys, indexes = [], [], []
    for symbol, group in frame.groupby('symbol', sort=False, observed=True):
        group = group.sort_index()
        if len(group) < lookback:
            log(f'Skipping {symbol} sequences: only {len(group):,} rows in this split.')
            continue
        arrays = builder.build(group, feature_columns=feature_columns)
        if len(arrays.y):
            xs.append(arrays.x)
            ys.append(arrays.y)
            indexes.append(arrays.index)
    if not xs:
        raise RuntimeError('No sequence arrays could be built for this split.')
    index = pd.DatetimeIndex(np.concatenate([idx.to_numpy() for idx in indexes]))
    return SequenceDatasetArrays(
        x=np.concatenate(xs, axis=0),
        y=np.concatenate(ys, axis=0),
        index=index,
    )


def shuffle_sequences(arrays, *, seed):
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(arrays.y))
    return SequenceDatasetArrays(
        x=arrays.x[order],
        y=arrays.y[order],
        index=pd.DatetimeIndex(arrays.index[order]),
    )


def count_symbol_sequences(frame, *, lookback):
    total = 0
    for _, group in frame.groupby('symbol', sort=False, observed=True):
        if len(group) >= lookback:
            total += len(group) - lookback + 1
    return total


log('Preparing processed feature source...')
processed_root = DATA_DIR / 'processed'
if STAGE_PROCESSED_TO_LOCAL:
    processed_root = stage_processed_feature_files(
        DATA_DIR / 'processed',
        LOCAL_PROCESSED_DIR,
        pattern='*_5m_features.parquet',
        force_refresh=FORCE_REFRESH_LOCAL_STAGE,
        limit=MAX_FILES or None,
    )
else:
    log('Local processed staging is disabled; reading processed Parquets from Drive.')
    if CLEAN_LOCAL_STAGE_WHEN_DISABLED and LOCAL_PROCESSED_DIR.exists():
        shutil.rmtree(LOCAL_PROCESSED_DIR)
        log(f'Removed old local staged features at {LOCAL_PROCESSED_DIR}')

log_resources('After processed source preparation')
log(f'Loading instruments from {processed_root}...')
instruments = load_instruments(
    processed_root,
    pattern='*_5m_features.parquet',
    min_rows=MIN_ROWS,
    limit=None if STAGE_PROCESSED_TO_LOCAL else (MAX_FILES or None),
)
log_resources('After loading instruments')

if len(instruments) < 2:
    log('Need at least two usable 5-minute feature Parquet files for combined training.')
else:
    log('Selecting shared feature columns...')
    split_config = SplitConfig()
    base_feature_columns = shared_feature_columns(instruments)
    log_resources('After selecting shared feature columns')
    if not base_feature_columns:
        raise RuntimeError('No shared numeric feature columns found across processed files.')

    log('Combining instruments and creating calendar-date split...')
    combined = combine_instruments(instruments, feature_columns=base_feature_columns)
    combined_rows = len(combined)
    splits = date_based_split(combined, split_config)
    del combined
    log_resources('After combining and splitting')
    symbols = [instrument.name for instrument in instruments]
    log(
        f"Combined rows={combined_rows:,} symbols={len(symbols):,} shared_features={len(base_feature_columns):,} "
        f"train_end={splits.train_end_date.date()} val_end={splits.validation_end_date.date()}"
    )
    log(
        f"split rows: train={len(splits.train):,} validation={len(splits.validation):,} test={len(splits.test):,} "
        f"device={device}"
    )

    lgbm_result = None
    if TRAIN_LIGHTGBM:
        log('Starting pooled LightGBM training...')
        lgbm_result = train_pooled_lgbm_from_splits(
            splits,
            instruments=instruments,
            feature_columns=base_feature_columns,
            paths=paths,
            output_name=SCOPE,
            split_config=split_config,
            include_symbol_feature=True,
            lightgbm_device_type=LGBM_DEVICE_TYPE,
            lightgbm_gpu_device_id=LGBM_GPU_DEVICE_ID,
        )
        log(f"lgbm_model: {lgbm_result['model_path']}")
        log(f"lgbm_validation: {lgbm_result['metrics']['validation']}")
        log(f"lgbm_test: {lgbm_result['metrics']['test']}")
        log_resources('After LightGBM training')

    train_frame = splits.train
    validation_frame = splits.validation
    train_end_date = splits.train_end_date
    validation_end_date = splits.validation_end_date
    n_train_rows = len(train_frame)
    n_validation_rows = len(validation_frame)
    n_test_rows = len(splits.test)
    del splits
    del instruments
    gc.collect()
    log_resources('After releasing per-instrument frames and test split')

    effective_num_workers = NUM_WORKERS if device.type == 'cuda' else 0
    log('Preparing lazy per-symbol sequence datasets for LSTM/GRU...')
    input_size = len(base_feature_columns)
    n_train_sequences = count_symbol_sequences(train_frame, lookback=LOOKBACK)
    n_validation_sequences = count_symbol_sequences(validation_frame, lookback=LOOKBACK)
    log_resources('After preparing lazy sequence datasets')
    log(
        f"sequence rows: train={n_train_sequences:,} validation={n_validation_sequences:,} "
        f"input_size={input_size:,}"
    )

    torch_results = {}
    if TRAIN_LSTM:
        log('Starting LSTM training...')
        lstm = LSTMClassifier(input_size=input_size, hidden_size_1=128, hidden_size_2=64, dropout=0.3)
        lstm_result = train_torch_classifier_from_frames(
            model=lstm,
            model_name=f'lstm_{SCOPE}',
            train_frame=train_frame,
            validation_frame=validation_frame,
            feature_columns=base_feature_columns,
            lookback=LOOKBACK,
            model_dir=MODEL_DIR,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            num_workers=effective_num_workers,
            device=device,
            log_every_batches=LOG_EVERY_BATCHES,
        )
        torch_results['lstm'] = result_to_dict(lstm_result)
        log(f"lstm_final: {torch_results['lstm']['final_model']}")
        log_resources('After LSTM training')

    if TRAIN_GRU:
        log('Starting GRU training...')
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        gru = GRUClassifier(input_size=input_size, hidden_size=128, dropout=0.3)
        gru_result = train_torch_classifier_from_frames(
            model=gru,
            model_name=f'gru_{SCOPE}',
            train_frame=train_frame,
            validation_frame=validation_frame,
            feature_columns=base_feature_columns,
            lookback=LOOKBACK,
            model_dir=MODEL_DIR,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            num_workers=effective_num_workers,
            device=device,
            log_every_batches=LOG_EVERY_BATCHES,
        )
        torch_results['gru'] = result_to_dict(gru_result)
        log(f"gru_final: {torch_results['gru']['final_model']}")
        log_resources('After GRU training')

    combined_metadata_path = MODEL_DIR / f'training_{SCOPE}_metadata.json'
    combined_metadata = {
        'scope': SCOPE,
        'symbols': symbols,
        'interval': '5m',
        'train_end_date': str(train_end_date.date()),
        'validation_end_date': str(validation_end_date.date()),
        'n_train_rows': n_train_rows,
        'n_validation_rows': n_validation_rows,
        'n_test_rows': n_test_rows,
        'n_train_sequences': n_train_sequences,
        'n_validation_sequences': n_validation_sequences,
        'feature_columns': base_feature_columns,
        'lgbm_feature_columns': base_feature_columns + ['symbol'],
        'lgbm_model': str(lgbm_result['model_path']) if lgbm_result else None,
        'lgbm_metrics': str(lgbm_result['metrics_path']) if lgbm_result else None,
        'lstm_model': torch_results.get('lstm', {}).get('final_model'),
        'gru_model': torch_results.get('gru', {}).get('final_model'),
        'device': str(device),
        **torch_results,
    }
    write_torch_training_metadata(combined_metadata_path, combined_metadata)
    log(f'combined_metadata: {combined_metadata_path}')
    log('Completed combined NIFTY50 training.')


## 4. Run order summary

For a full training run when all parquets are already built on Drive:

1. Run the **Setup** cell (Drive mount + pip install)
2. Run the **Git pull** cell
3. Run the **LightGBM install** cell — confirm the CUDA smoke test passes
4. Run the **Config** cell
5. Skip cells 1 and 2 (`RUN_RAW_CACHE = False`, `RUN_FEATURE_BUILD = False`)
6. Run **Cell 3** — full combined training
7. Run the **Archive** cell after training completes

For a fresh setup (no parquets yet): set `RUN_RAW_CACHE = True` in cell 1 and `RUN_FEATURE_BUILD = True` in cell 2, run all cells top to bottom.

## 5. Archive outputs

Create one timestamped archive in Drive containing trained models, reports, and artifacts.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import zipfile

ARCHIVE_DIR = BASE / 'archives'
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
archive_path = ARCHIVE_DIR / f'tradingbot26_outputs_{timestamp}.zip'

include_dirs = [MODEL_DIR, REPORT_DIR, ARTIFACT_DIR]
with zipfile.ZipFile(archive_path, mode='w', compression=zipfile.ZIP_DEFLATED) as archive:
    for directory in include_dirs:
        if not directory.exists():
            print(f'Skipping missing directory: {directory}')
            continue
        for path in directory.rglob('*'):
            if path.is_file():
                archive.write(path, arcname=path.relative_to(BASE))

size_mb = archive_path.stat().st_size / (1024 * 1024)
print(f'Archive written: {archive_path}')
print(f'Size: {size_mb:.2f} MB')